# Diffusion

Diffusion 是近几年用的比较多的生成模型，其数学表示也很完善。对比其余主流的生成模型，如自回归，在视觉自回归工作（VAR）之前，Diffusion 的生成效率更高，因为它每轮处理可以一次性处理图片的所有 token。对比如 PixelCNN 等需要逐 token 生成的方式，效率高很多。

关于 Diffusion 的具体推导，网上已经有很多教程，这里列几个我觉得不错的：

1. [苏剑林科学空间](https://spaces.ac.cn/archives/9119)
2. [用大白话讲解 Diffusion](https://zhuanlan.zhihu.com/p/610012156)
3. [Diffusion 公式推导](https://zhuanlan.zhihu.com/p/1937935753352053597)

## Diffusion 的整体思路

Diffusion 的思路，是将图像生成过程视为从噪声中逐步恢复的过程。这一过程包括两部分：前向加噪和反向去噪。

## 前向加噪

其中，前向加噪可以表示如下：

$$
x_t = \sqrt{\alpha_t}x_{t-1} + \sqrt{\beta_t}\epsilon
$$

其中：

$$
\beta_t + \alpha_t = 1, \qquad \epsilon \sim \mathcal{N}(0, I)
$$

这里的 $\epsilon$ 就是上面说的噪声。关于 $\alpha / \beta$ 的设置，可以直观理解为，随着加噪过程的进行，噪声的影响是越来越弱的。就像往糖水里加糖，当水越来越甜，需要加更多糖效果才够。因此 $\beta$ 随着 $t$ 增大在逐渐增大，$\alpha$ 则逐渐减小。当然更底层原因笔者没有去探究，欢迎补充。

## 加噪公式的并行化

在实际应用中，我们需要考虑到模型的效率，在计算机领域提升效率最直接的手段就是提高并行程度。在上述公式中，我们要知道第 $t$ 轮加噪的结果，需要首先有 $t-1$ 时刻的 $x_{t-1}$，这个过程是串行的。

但是，实际上我们可以通过公式简化。这里具体过程参考 [Diffusion 公式推导](https://zhuanlan.zhihu.com/p/1937935753352053597)，总之简化后的表示为：

$$
x_t = \sqrt{\hat{\alpha}_t}x_0 + \sqrt{\hat{\beta}_t}\epsilon
$$

其中：

$$
\hat{\alpha}_t = \alpha_1 \alpha_2 \cdots \alpha_t
$$

$\hat{\beta}_t$ 同理。

我们生成图像的目标，实际上是想要知道在 $x_t$ 的先验上，$x_0$ 的后验分布，即求：

$$
q(x_t \mid x_0)
$$

这个时候可能有人会问了，我都有 $x_t$ 和 $x_0$ 的关系式了，那不是可以直接求 $x_0$，这里噪声用一个神经网络预测就行，为什么还要逐步预测。这里抛开定性的解释，我的理解是直接由 $x_t$ 到 $x_0$ 的噪声方差很大（$\hat{\beta}_t$），方差大意味着拟合更困难，解更多。因此一步一步来去噪声，会更好。

## 反向去噪

加噪过程已经清楚了，那么接下来是去噪声过程，这里神经网络就需要登场了。对于这部分的推导，有两种理解方式。

一种是像 [用大白话讲解 Diffusion](https://zhuanlan.zhihu.com/p/610012156) 一样，直接求出：

$$
q(x_{t-1} \mid x_t, x_0)
$$

得到其是一个正态分布，均值与 $x_0$ 和 $x_t$ 和噪声相关。其中 $x_0$ 可以有任意 $x_t$ 反推得到，最后未知量就是一个噪声，均值表示为：

$$
u_t = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\hat{\beta}_t}\epsilon\right)
$$

然后用神经网络拟合这个噪声。我们实际得到的是每一次去噪声的期望值。

另一种理解方式是，我们有 $x_t$ 与 $x_{t-1}$ 的关系，那可以直接：

$$
x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \beta_t\epsilon\right)
$$

这与前面实际上就差了个系数，相当于少给了一些先验。怎么由这个公式去得到与前面一致的表达式，可以参考 [苏剑林科学空间](https://spaces.ac.cn/archives/9119)，总体思路就是，在训练时，这里的 $x_t$ 是 teacher-force 的 $x_t$，即 ground truth 的 $x_t$，而非去噪后得到的 $x_t$，其值是由 $x_0$ 得到的。而直接带入 $x_t$ 公式中会出现两个噪声，多次采样噪声会增加训练的不稳定性。因此做了额外的处理。


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Diffusion(nn.Module):
    def __init__(
        self,
        model,
        num_steps=1000,
        beta_start=1e-4,
        beta_end=0.02,
    ):
        super().__init__()

        self.model = model
        self.num_steps = num_steps

        # beta_t
        betas = torch.linspace(
            beta_start,
            beta_end,
            num_steps,
            dtype=torch.float32
        )

        # alpha_t = 1 - beta_t
        alphas = 1.0 - betas

        # alpha_bar_t = alpha_1 * ... * alpha_t
        alpha_cumprod = torch.cumprod(alphas, dim=0)

        # 注册成 buffer：
        # 不参与训练，但是会自动跟随 model.to(device)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer(
            "alpha_cumprod",
            alpha_cumprod
        )

        self.register_buffer(
            "sqrt_alpha_cumprod",
            torch.sqrt(alpha_cumprod)
        )

        self.register_buffer(
            "sqrt_one_minus_alpha_cumprod",
            torch.sqrt(1.0 - alpha_cumprod)
        )

    def add_noise(self, x0, t, noise=None):
        """
        x0:
            [B, C, H, W]

        t:
            [B]

        noise:
            [B, C, H, W]

        return:
            x_t
        """

        if noise is None:
            noise = torch.randn_like(x0)

        sqrt_alpha_bar = self.sqrt_alpha_cumprod[t]
        sqrt_alpha_bar = sqrt_alpha_bar.view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar = (
            self.sqrt_one_minus_alpha_cumprod[t]
        )
        sqrt_one_minus_alpha_bar = (
            sqrt_one_minus_alpha_bar.view(-1, 1, 1, 1)
        )

        # DDPM forward process:
        #
        # x_t =
        # sqrt(alpha_bar_t) * x_0
        # +
        # sqrt(1 - alpha_bar_t) * epsilon
        #
        xt = (
            sqrt_alpha_bar * x0
            + sqrt_one_minus_alpha_bar * noise
        )

        return xt, noise

    def forward(self, x0, t=None):
        batch_size = x0.shape[0]

        if t is None:
            t = torch.randint(
                0,
                self.num_steps,
                (batch_size,),
                device=x0.device,
                dtype=torch.long
            )

        # epsilon ~ N(0, I)
        noise = torch.randn_like(x0)

        xt, noise = self.add_noise(
            x0,
            t,
            noise
        )

        # epsilon_theta(x_t, t)
        predicted_noise = self.model(
            xt,
            t
        )

        # simple DDPM loss
        loss = F.mse_loss(
            predicted_noise,
            noise
        )

        return loss